# FAKE NEWS GROUP PROJECT: Group 54
## Preparing for the code

In [ ]:
# Importing the needed modules
import re
import math
import nltk
import time
import pandas as pd
from scipy.sparse import hstack
import matplotlib.pyplot as plt
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
# Only needed once
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\bruge\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bruge\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\bruge\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bruge\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Part 1 - Task 1

In [3]:
# Loading the dataset
data = pd.read_csv('fakenews_sample.csv')
data.head()

,Unnamed: 0,id,domain,type,url,content,scraped_at,inserted_at,updated_at,title,authors,keywords,meta_keywords,meta_description,tags,summary
0,0,141,awm.com,unreliable,http://awm.com/church-congregation-brings-gift...,Sometimes the power of Christmas will make you...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Church Congregation Brings Gift to Waitresses ...,Ruth Harris,NaN,[''],NaN,NaN,NaN
1,1,256,beforeitsnews.com,fake,http://beforeitsnews.com/awakening-start-here/...,AWAKENING OF 12 STRANDS of DNA – “Reconnecting...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,AWAKENING OF 12 STRANDS of DNA – “Reconnecting...,Zurich Times,NaN,[''],NaN,NaN,NaN
2,2,700,cnnnext.com,unreliable,http://www.cnnnext.com/video/18526/never-hike-...,Never Hike Alone: A Friday the 13th Fan Film U...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Never Hike Alone - A Friday the 13th Fan Film ...,NaN,NaN,[''],Never Hike Alone: A Friday the 13th Fan Film ...,NaN,NaN
3,3,768,awm.com,unreliable,http://awm.com/elusive-alien-of-the-sea-caught...,"When a rare shark was caught, scientists were ...",2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Elusive ‘Alien Of The Sea ‘ Caught By Scientis...,Alexander Smith,NaN,[''],NaN,NaN,NaN
4,4,791,bipartisanreport.com,clickbait,http://bipartisanreport.com/2018/01/21/trumps-...,Donald Trump has the unnerving ability to abil...,2018-01-25 16:17:44.789555,2018-02-02 01:19:41.756632,2018-02-02 01:19:41.756664,Trump’s Genius Poll Is Complete & The Results ...,Gloria Christie,NaN,[''],NaN,NaN,NaN


In [4]:
# Function to clean the dataset
def clean_text(text):
    # Empty values
    if pd.isna(text):
        return ""
    
    # lower the text
    text = text.lower()

    # Replace url with a <URL> tag
    text = re.sub(r'https?://\S+|www\.\S+', '<URL>', text)

    # Replace emails with <EMAIL> tag
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '<EMAIL>', text)

    # Replace dates with <DATE> tag
    text = re.sub(r'[0-9]+[a-zA-Z]+', '<DATE>', text)

    # Replace all other numbers with <NUM> tag
    text = re.sub(r'[0-9]+', '<NUM>', text)
    
    # Remove any special charachters
    text = re.sub(r'[^a-z\s<>]', '', text)

    # Remove spaces, tabs and line shifts
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text # Returning the cleaned text

In [5]:
# Applying the cleaning function to the content column of the dataset
data["clean_content"] = data["content"].apply(clean_text)
data[["content", "clean_content"]].head()

# Tokenize
data['tokens'] = data['clean_content'].apply(nltk.word_tokenize)
data[['clean_content', 'tokens']].head()

# Filter tokens 
english_stopwords = set(stopwords.words('english')) # Define the english stopwords
data['filtered_tokens'] = data['tokens'].apply(lambda tokens: [word for word in tokens if word not in english_stopwords])
data[['tokens', 'filtered_tokens']].head()

# Stemming
ps = PorterStemmer() # Creating a stemmer
data['stemmed'] = data['filtered_tokens'].apply(lambda x: [ps.stem(w) if not w.startswith('<') else w for w in x])

In [6]:
# Extracting all words from the tokens, filtered tokens and stemmed lists
all_words_list = [word for sublist in data['tokens'] for word in sublist]
filtered_words_list = [word for sublist in data['filtered_tokens'] for word in sublist]
all_stemmed_words = [word for sublist in data['stemmed'] for word in sublist]

# Finding unique words to create vocabularies
vocab = set(all_words_list)
vocab_filt = set(filtered_words_list)
vocab_stemmed = set(all_stemmed_words)

# Calculate lenght of the vocabularies
vocab_size = len(vocab)
filt_vocab_size = len(vocab_filt)
vocab_stemmed_size = len(vocab_stemmed)

In [ ]:
# Calculating the reduction in vocabulary size after removing stopwords
reduction_filtering = (1 - (filt_vocab_size / vocab_size)) * 100

# Calculating the reduction in vocabulary size after stemming
reduction_stemming = (1 - (vocab_stemmed_size / filt_vocab_size)) * 100

# Printing the statistics
print(f"Number of stopwords: {len(english_stopwords)}")
print(f"The first 15 english stopwords: {list(english_stopwords)[:15]}")
print("-------------------------------------")
print(f"Original Vocabulary Size: {vocab_size}")
print(f"Vocabulary Size after removing Stopwords: {filt_vocab_size} (Reduction: {reduction_filtering:.2f}%)")
print(f"Vocabulary Size after Stemming: {vocab_stemmed_size} (Reduction: {reduction_stemming:.2f}%)")

Number of stopwords: 198
The first 15 english stopwords: ['again', "didn't", 'his', "doesn't", 'ourselves', 'do', "it'll", 'which', 'own', 'about', 'ma', 'other', "it'd", 'from', 'had']
-------------------------------------
Original Vocabulary Size: 16471
Vocabulary Size after removing Stopwords: 16339 (Reduktion: 0.80%)
Vocabulary Size after Stemming: 10926 (Reduktion: 33.13%)


## Part 1 - Task 2

In [ ]:
# Create variable for input file
input_file = "995K_rows.csv"

# Define chunk size
chunk_size = 15000 

# Create a dataset reader
reader = pd.read_csv(input_file, chunksize=chunk_size)

# Create variable for output file
output_file = "processed_995K.csv"

# Initialize global statistics
global_vocab_initial = set()
global_vocab_filtered = set()
global_vocab_stemmed = set()

In [ ]:
# Print to keep track of the loading of the dataset
print(f"Starter processering af {input_file}...")
print(f"Chunk size: {chunk_size}")
print("------------------------------------------")

# Set the start time
start_time = time.time()

for i, chunk in enumerate(reader):
    chunk_start = time.time()
    
    # Apply the cleaning function
    chunk['content_cleaned'] = chunk['content'].apply(clean_text)

    # Tokenize
    chunk['tokens'] = chunk['content_cleaned'].apply(nltk.word_tokenize)
    
    # Update the original vocabulary
    for t_list in chunk['tokens']:
        global_vocab_initial.update(t_list)
    
    # Filter for stopwords
    chunk['filtered_tokens'] = chunk['tokens'].apply(lambda t: [w for w in t if w not in english_stopwords])

    # Update the filtered vocbulary
    for t_list in chunk['filtered_tokens']:
        global_vocab_filtered.update(t_list)
    
    # Stemming
    chunk['stemmed'] = chunk['filtered_tokens'].apply(lambda x: [ps.stem(w) if not w.startswith('<') else w for w in x])

    # Update the stemmed vocabulary
    for t_list in chunk['stemmed']:
        global_vocab_stemmed.update(t_list)
    
    # Save the data to the output file
    is_first = (i == 0)
    chunk.to_csv(output_file, index=False, mode='w' if is_first else 'a', header=is_first)
    
    # Calculate the status
    chunk_duration = time.time() - chunk_start
    total_elapsed = time.time() - start_time
    rows_so_far = (i + 1) * chunk_size
    
    # Print the status to keep track of process
    print(f"Chunk {i+1:3} | Rækker: {rows_so_far:7,} | Tid: {chunk_duration:5.2f}s | Total: {total_elapsed/60:5.2f} min")

In [ ]:
# Find lenght of the datasets
vocab_og = len(global_vocab_initial)
vocab_filt = len(global_vocab_filtered)
vocab_stem = len(global_vocab_stemmed)

# Calculate reduction in vocabulary size
red_filt = (1 - (vocab_filt/vocab_og)) * 100
red_stem = (1 - (vocab_stem/vocab_og)) * 100

# Printing the statistics
print(f"Original Vocabulary Size: {vocab_og}")
print(f"Vocabulary Size after removing Stopwords: {vocab_filt} (Reduction: {red_filt:.2f}%)")
print(f"Vocabulary Size after Stemming: {vocab_stem} (Reduction: {red_stem:.2f}%)")

## Part 1 - Task 3

## Part 2 - Task 1